In [1]:
import numpy as np
import networkx as nx
from qiskit.quantum_info import SparsePauliOp
from qiskit_qaoa.utils.sat_mapper import HigherOrderSatMapper

from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate, CXGate

from qiskit.transpiler import PassManager
# from qiskit.transpiler.passes import HighLevelSynthesis, InverseCancellation
# from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping


from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, CommutingGateRouter, FindCommutingPauliEvolutionsMulti, DecomposePauliZEvolution

In [ ]:
extended_swap_strat = ExtendedSwapStrategy.from_heavy_hex(1, 2)
num_physical_qubits = extended_swap_strat._num_vertices
num_physical_qubits

21

In [3]:
n = 3
T = 3
num_qubits = n*T
rng = np.random.default_rng(seed=10)
doubles = rng.choice(num_qubits, (20, 2))
quads = rng.choice(num_qubits, (2, 4), replace=False)
sixes = rng.choice(num_qubits, (1, 6), replace=False)

coeffs = rng.random(23)

def choice_to_pauli(choice: list[int], size: int) -> str:
    pauli = ['I'] * size
    for x in choice:
        pauli[size - x - 1] = 'Z'
    return ''.join(pauli)

hamiltonian = SparsePauliOp(
    [choice_to_pauli(c, num_qubits) for c in doubles] + [choice_to_pauli(c, num_qubits) for c in quads] + [choice_to_pauli(c, num_qubits) for c in sixes], 
    coeffs=coeffs
)
hamiltonian = hamiltonian.simplify()
hamiltonian = hamiltonian.sort(weight=True)

In [4]:
def hamiltonian_to_interactions(hamiltonian: SparsePauliOp) -> list[tuple]:
    interactions = []
    for t in hamiltonian:
        if np.sum(t.paulis[0].z) < 2 or np.sum(t.paulis[0].z) > 6:
            continue
        if np.sum(t.paulis[0].z) == 2:
            edge = np.nonzero(t.paulis[0].z)[0]
            interactions.append(edge)
        if np.sum(t.paulis[0].z) == 4 or np.sum(t.paulis[0].z) == 6:
            edge = np.nonzero(t.paulis[0].z)[0]
            interactions.append(edge)
    return interactions

In [5]:
program_interactions = hamiltonian_to_interactions(hamiltonian)

In [6]:
program_interactions

[array([1, 2]),
 array([0, 3]),
 array([1, 4]),
 array([3, 6]),
 array([5, 6]),
 array([1, 7]),
 array([2, 7]),
 array([3, 7]),
 array([4, 7]),
 array([1, 8]),
 array([3, 8]),
 array([4, 8]),
 array([6, 8]),
 array([7, 8]),
 array([1, 2, 3, 7]),
 array([0, 4, 5, 8]),
 array([0, 2, 4, 5, 7, 8])]

In [7]:
mapper = HigherOrderSatMapper(timeout=60)
results = {}
num_layers = 0
sat_results = mapper.hubo_max_sat(
    program_interactions, extended_swap_strat, num_layers
)
mapping = sat_results[num_layers][1]
edge_map = dict(mapping)
print(f'Cost: {sat_results[num_layers][0]}')
print(edge_map)

15:38:06 - qiskit_qaoa.utils.sat_mapper - INFO - Num layers: 0
15:38:06 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
15:38:06 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
15:38:11 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
15:39:53 - qiskit_qaoa.utils.sat_mapper - INFO - Hard constraints: 2655
15:39:53 - qiskit_qaoa.utils.sat_mapper - INFO - Soft constraints: 4102917
Cost: 8
{0: 2, 1: 8, 2: 19, 3: 14, 4: 18, 5: 16, 6: 9, 7: 7, 8: 20}


In [ ]:
edge_map = {0: 2, 1: 8, 2: 19, 3: 14, 4: 18, 5: 16, 6: 9, 7: 7, 8: 20}

In [ ]:
sorted(doubles, key=lambda e: 10*e[0]+e[1])

In [ ]:
quads

In [ ]:
clause = np.array([-41, 76, 88, 94])
print( (np.abs(clause) -1) // 35)
print( ((np.abs(clause) - 1) % 35) * np.sign(clause))

In [ ]:
clause = np.array([-41, 146, 158, 164])
print( (np.abs(clause) -1) // 35)
print( ((np.abs(clause) - 1) % 35) * np.sign(clause))

In [ ]:
clause = np.array([-132, 253, 254, 272])
print( (np.abs(clause) -1) // 35)
print( ((np.abs(clause) - 1) % 35) * np.sign(clause))

In [ ]:
clause = np.array([-161, 248, 253, 266])
print( (np.abs(clause) -1) // 35)
print( ((np.abs(clause) - 1) % 35) * np.sign(clause))

In [ ]:
clause = np.array([-270, 287, 288, 305])
print( (np.abs(clause) -1) // 35)
print( ((np.abs(clause) - 1) % 35) * np.sign(clause))

In [ ]:
clause = np.array([-18, -41, -217, 269])
print( (np.abs(clause) -1) // 35)
print( ((np.abs(clause) - 1) % 35) * np.sign(clause))

In [ ]:
clause = np.array([-29, -159, -185])
print( (np.abs(clause) -1) // 35)
print( ((np.abs(clause) - 1) % 35) * np.sign(clause))

In [8]:
new_hamiltonian = hamiltonian.apply_layout([edge_map[i] for i in range(num_qubits)], num_physical_qubits)

In [9]:
pm = PassManager(
    [
        # HighLevelSynthesis(basis_gates=["PauliEvolution"]), # Not needed if set up circuit as PauliEvolutionGate
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouter(
            extended_swap_strat,
            max_layers=num_layers,
            perform_extra_swaps=False
        ),
        # SwapToFinalMapping(),
        # DecomposePauliZEvolution(extended_swap_strat._coupling_map),
        # # HighLevelSynthesis(
        # #     basis_gates=["sx", "x", "rz", "rzz", "cx", "id", "swap"], 
        # #     coupling_map=extended_swap_strat._coupling_map, 
        # #     use_qubit_indices=True,
        # #     qubits_initially_zero=False
        # # ),
        # InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [10]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(new_hamiltonian), range(num_physical_qubits))
tqc = pm.run(qc)

15:42:22 - qiskit_qaoa.utils.transpiler_passes - INFO - Max layers needed to apply swap decompose: 0
15:42:23 - qiskit_qaoa.utils.transpiler_passes - INFO - Gates we cannot directly implement: 8
15:42:23 - qiskit_qaoa.utils.transpiler_passes - INFO - [(14, 20), (9, 14), (2, 7, 16, 18, 19, 20), (18, 20), (2, 16, 18, 20), (7, 8), (7, 20), (8, 18)]
15:42:23 - qiskit_qaoa.utils.transpiler_passes - INFO - Not implementing those gates


In [ ]:
print(
    (edge_map[1], edge_map[2]), (edge_map[1], edge_map[4]),(edge_map[3], edge_map[7]),
    (edge_map[4], edge_map[7]),(edge_map[7], edge_map[8]),
    (edge_map[0], edge_map[1], edge_map[6], edge_map[7]),
    (edge_map[2], edge_map[3], edge_map[5], edge_map[8]),
)